# 02_kassel_exploration.ipynb

## Σκοπός

Αυτό το notebook αποτελεί το **strict raw decoding and timestamp integrity validation** στάδιο για το DaKS / Kassel dataset.

Ελέγχει αποκλειστικά:
- το raw input / target file pairing,
- τα **explicit allowed timestamp formats** ανά file type,
- το separator handling,
- την temporal ordering integrity ανά park,
- και την **input-target merge integrity** πάνω στο timestamp.

Δεν καλύπτει:
- modeling,
- train / validation / test splitting,
- feature engineering,
- future-work additions.

## Methodological note

Η λογική είναι σκόπιμα αυστηρή:
- χρησιμοποιούνται **ρητά επιτρεπτά** timestamp formats ανά file type,
- αποφεύγονται ambiguous mixed parsing heuristics,
- και κάθε park αξιολογείται με interpretable failure logging.

In [23]:
# Standard library imports for file-system access and filename pattern matching
import re
from pathlib import Path

# Third-party imports used for tabular auditing and notebook display
import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import display

# Canonical raw dataset location expected by the repository structure
RAW_DIR = Path("../data/raw/kassel_dataset")

# Fail early if the expected raw data directory is missing
assert RAW_DIR.exists(), f"Raw data directory not found: {RAW_DIR.resolve()}"

# Collect raw CSV files while excluding the metadata file from input-target pair auditing
all_csv_files = sorted(
    [p for p in RAW_DIR.glob("*.csv") if p.name.lower() != "meta.csv"]
)

print(f"Raw CSV files found: {len(all_csv_files)}")
print(f"Raw directory: {RAW_DIR.resolve()}")

Raw CSV files found: 544
Raw directory: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\raw\kassel_dataset


In [24]:
# Strict filename pattern for raw park-level input and target files
# Expected examples:
# - data_input_00001.csv
# - data_target_00001.csv
PAIR_PATTERN = re.compile(r"^data_(input|target)_(\d+)\.csv$", re.IGNORECASE)

# Dictionaries keyed by zero-padded park ID
input_files = {}
target_files = {}

# Files that do not follow the strict expected naming convention are logged separately
unmatched_files = []

# Index all raw CSV files into input/target groups
for path in all_csv_files:
    match = PAIR_PATTERN.match(path.name)

    if not match:
        unmatched_files.append(path.name)
        continue

    file_type, park_id = match.groups()
    park_id = park_id.zfill(5)

    if file_type.lower() == "input":
        input_files[park_id] = path
    else:
        target_files[park_id] = path

# Build global park ID sets for complete pairing diagnostics
all_input_ids = set(input_files.keys())
all_target_ids = set(target_files.keys())
all_park_ids = sorted(all_input_ids | all_target_ids)
paired_park_ids = sorted(all_input_ids & all_target_ids)

# Create a park-level pairing audit so missing pairs are explicitly reported
pair_audit_rows = []

for park_id in all_park_ids:
    has_input = park_id in input_files
    has_target = park_id in target_files

    if has_input and has_target:
        pair_status = "paired"
    elif has_input and not has_target:
        pair_status = "missing_target"
    elif has_target and not has_input:
        pair_status = "missing_input"
    else:
        pair_status = "unexpected"

    pair_audit_rows.append(
        {
            "park_id": park_id,
            "pair_status": pair_status,
            "input_file": input_files.get(park_id, Path("")).name or None,
            "target_file": target_files.get(park_id, Path("")).name or None,
        }
    )

pair_audit_df = pd.DataFrame(pair_audit_rows)

print(f"Input files indexed : {len(all_input_ids)}")
print(f"Target files indexed: {len(all_target_ids)}")
print(f"Unique park IDs     : {len(all_park_ids)}")
print(f"Paired parks        : {len(paired_park_ids)}")

if unmatched_files:
    print("\nFiles ignored by the strict pair pattern:")
    print(unmatched_files[:10])

display(
    pair_audit_df["pair_status"]
    .value_counts(dropna=False)
    .rename_axis("pair_status")
    .reset_index(name="parks")
)

# The notebook must have at least one valid input-target pair to proceed
assert len(paired_park_ids) > 0, "No valid input-target park pairs were found."

Input files indexed : 272
Target files indexed: 272
Unique park IDs     : 272
Paired parks        : 272


,pair_status,parks
0,paired,272


## Strict parsing policy

Σε αυτό το στάδιο ορίζονται **ρητά** τα επιτρεπτά timestamp formats ανά file type.

- **Input files**: επιτρέπονται μόνο τα formats που έχουν ήδη εγκριθεί από το raw inspection policy.
- **Target files**: επιτρέπεται μόνο το canonical target format.
- Δεν χρησιμοποιούνται `dayfirst=True`, `infer_datetime_format`, ή mixed parsing heuristics.
- Κάθε αρχείο πρέπει να είναι internally consistent ως προς το timestamp format του.

In [25]:
# Explicit timestamp formats accepted by the raw audit.
# These are intentionally restrictive to avoid ambiguous parsing behavior.
INPUT_TS_FORMATS = [
    "%d/%m/%Y %H:%M",      # Example: 8/12/2018 0:00
    "%Y-%m-%d %H:%M:%S",   # Example: 2018-12-08 00:00:00
]

TARGET_TS_FORMATS = [
    "%Y-%m-%d %H:%M:%S",   # Example: 2018-12-08 00:00:00
]


def choose_separator(file_path: Path, file_type: str):
    """
    Read a raw CSV with candidate separators and retain the most plausible parse.

    Rationale:
    - Raw files may use different separators depending on file type.
    - The function evaluates a small set of allowed separators and scores the result.
    - A parse that collapses the file into a single column is treated as implausible.
    """
    separator_candidates = [";", ","]
    time_candidates = {"fcst_time", "time", "timestamp"}

    # File-type-specific value columns improve separator scoring.
    if file_type == "input":
        value_candidates = {"nwp_fcst_horiz_hours"}
        expected_sep = ";"
    else:
        value_candidates = {"pw", "icon_eu_daf_pc_baseline", "test_flag"}
        expected_sep = ","

    scored_reads = []

    for sep in separator_candidates:
        # Read as string to preserve raw values before explicit downstream validation.
        df = pd.read_csv(
            file_path,
            sep=sep,
            dtype="string",
            keep_default_na=False,
        )

        # Normalize column names for robust downstream matching.
        df.columns = [str(col).strip() for col in df.columns]

        # Score the parse using lightweight structural heuristics.
        score = 0
        score += len(df.columns)

        # A one-column parse usually indicates a wrong separator.
        if len(df.columns) == 1:
            score -= 100

        # Reward presence of plausible timestamp columns.
        if any(col in df.columns for col in time_candidates):
            score += 50

        # Reward presence of plausible value columns.
        if any(col in df.columns for col in value_candidates):
            score += 10

        # Slight bias toward the expected separator for the given file type.
        if sep == expected_sep:
            score += 1

        scored_reads.append((score, sep, df))

    # Retain the highest-scoring parse.
    scored_reads.sort(key=lambda item: item[0], reverse=True)
    best_score, best_sep, best_df = scored_reads[0]

    # Hard failure if separator resolution still produces a degenerate parse.
    if len(best_df.columns) == 1:
        raise ValueError(
            f"Separator detection failed for {file_path.name}. "
            "Both candidate parses collapsed to a single column."
        )

    return best_df, best_sep


def get_time_column(df: pd.DataFrame, file_type: str, file_name: str) -> str:
    """
    Resolve the timestamp column using a strict file-type-specific search order.

    This keeps the schema logic explicit and prevents hidden assumptions.
    """
    if file_type == "input":
        candidates = ["fcst_time", "time", "timestamp"]
    else:
        candidates = ["time", "timestamp", "fcst_time"]

    for col in candidates:
        if col in df.columns:
            return col

    raise KeyError(
        f"No timestamp column found in {file_name}. "
        f"Available columns={df.columns.tolist()}"
    )


def parse_timestamp_strict(series: pd.Series, allowed_formats: list[str], field_name: str) -> pd.Series:
    """
    Parse timestamps using only the declared formats.

    Notes:
    - No mixed parsing heuristics are allowed here.
    - Empty strings are treated as missing before parsing.
    - The function fails if none of the approved formats can parse the series.
    """
    raw_str = series.astype("string").str.strip()
    empty_mask = raw_str.isna() | (raw_str == "")

    last_error = None

    for fmt in allowed_formats:
        try:
            parsed = pd.to_datetime(
                raw_str.mask(empty_mask),
                format=fmt,
                errors="raise",
            )
            return parsed
        except Exception as exc:
            last_error = exc

    sample_value = raw_str[~empty_mask].iloc[0] if (~empty_mask).any() else "<empty>"

    raise ValueError(
        f"Strict timestamp parsing failed for {field_name}. "
        f"Sample='{sample_value}' | Allowed formats={allowed_formats} | Last error={last_error}"
    )


def audit_time_series(ts: pd.Series) -> dict:
    """
    Summarize core timestamp integrity properties before canonical sorting.

    These diagnostics are used both for successful audits and for interpretable failure logging.
    """
    return {
        "row_count": int(len(ts)),
        "nat_count": int(ts.isna().sum()),
        "duplicate_count": int(ts.duplicated().sum()),
        "monotonic_before_sort": bool(ts.is_monotonic_increasing),
        "min_time": ts.min(),
        "max_time": ts.max(),
    }


def validate_exact_timestamp_alignment(input_ts: pd.Series, target_ts: pd.Series) -> dict:
    """
    Check whether input and target timestamps match exactly after canonical sorting.

    Method:
    - sort both timestamp series,
    - perform an outer merge on the timestamp key,
    - quantify unmatched timestamps on each side,
    - require exact one-to-one alignment.
    """
    input_key = pd.DataFrame({"timestamp": input_ts.sort_values().reset_index(drop=True)})
    target_key = pd.DataFrame({"timestamp": target_ts.sort_values().reset_index(drop=True)})

    outer_check = input_key.merge(
        target_key,
        on="timestamp",
        how="outer",
        indicator=True,
        validate="one_to_one",
    )

    left_only = int((outer_check["_merge"] == "left_only").sum())
    right_only = int((outer_check["_merge"] == "right_only").sum())
    both = int((outer_check["_merge"] == "both").sum())

    exact_match = (
        left_only == 0 and
        right_only == 0 and
        both == len(input_key) == len(target_key)
    )

    return {
        "left_only_count": left_only,
        "right_only_count": right_only,
        "matched_count": both,
        "exact_match": exact_match,
    }


def load_pair_raw(park_id: str) -> dict:
    """
    Load the raw input-target pair and resolve separator plus timestamp columns.

    This function performs only raw loading and schema resolution.
    Strict timestamp validation is handled separately.
    """
    input_path = input_files[park_id]
    target_path = target_files[park_id]

    df_input, input_sep_used = choose_separator(input_path, file_type="input")
    df_target, target_sep_used = choose_separator(target_path, file_type="target")

    input_time_col = get_time_column(df_input, file_type="input", file_name=input_path.name)
    target_time_col = get_time_column(df_target, file_type="target", file_name=target_path.name)

    return {
        "park_id": park_id,
        "input_path": input_path,
        "target_path": target_path,
        "df_input": df_input,
        "df_target": df_target,
        "input_sep_used": input_sep_used,
        "target_sep_used": target_sep_used,
        "input_time_col": input_time_col,
        "target_time_col": target_time_col,
    }


def validate_loaded_pair(raw_pair: dict) -> dict:
    """
    Run strict timestamp validation and exact one-to-one merge validation on a loaded pair.

    Validation steps:
    1. Parse timestamps with explicit allowed formats.
    2. Reject NaT values after parsing.
    3. Reject duplicate timestamps.
    4. Canonically sort by timestamp.
    5. Require exact input-target timestamp equality.
    6. Confirm one-to-one merge integrity.
    """
    df_input = raw_pair["df_input"].copy()
    df_target = raw_pair["df_target"].copy()

    input_time_col = raw_pair["input_time_col"]
    target_time_col = raw_pair["target_time_col"]

    # Parse timestamps using the explicit approved format list for each file type.
    df_input[input_time_col] = parse_timestamp_strict(
        df_input[input_time_col],
        INPUT_TS_FORMATS,
        field_name=f"{raw_pair['input_path'].name}:{input_time_col}",
    )
    df_target[target_time_col] = parse_timestamp_strict(
        df_target[target_time_col],
        TARGET_TS_FORMATS,
        field_name=f"{raw_pair['target_path'].name}:{target_time_col}",
    )

    # Collect temporal diagnostics prior to canonical sorting.
    input_audit = audit_time_series(df_input[input_time_col])
    target_audit = audit_time_series(df_target[target_time_col])

    # Reject missing timestamps after strict parsing.
    if input_audit["nat_count"] > 0:
        raise ValueError(f"Input contains {input_audit['nat_count']} NaT values after strict parsing.")
    if target_audit["nat_count"] > 0:
        raise ValueError(f"Target contains {target_audit['nat_count']} NaT values after strict parsing.")

    # Reject duplicate timestamps because downstream merge must be one-to-one.
    if input_audit["duplicate_count"] > 0:
        raise ValueError(f"Input contains {input_audit['duplicate_count']} duplicate timestamps.")
    if target_audit["duplicate_count"] > 0:
        raise ValueError(f"Target contains {target_audit['duplicate_count']} duplicate timestamps.")

    # Canonical sorting is applied only after raw ordering has already been audited.
    df_input = df_input.sort_values(input_time_col).reset_index(drop=True)
    df_target = df_target.sort_values(target_time_col).reset_index(drop=True)

    # Require exact timestamp equality between input and target after sorting.
    alignment = validate_exact_timestamp_alignment(
        df_input[input_time_col],
        df_target[target_time_col],
    )

    if not alignment["exact_match"]:
        raise ValueError(
            "Input-target timestamp mismatch detected. "
            f"left_only={alignment['left_only_count']}, "
            f"right_only={alignment['right_only_count']}, "
            f"matched={alignment['matched_count']}"
        )

    # Perform the final one-to-one merge only after alignment has been validated.
    merged_df = df_input.merge(
        df_target,
        left_on=input_time_col,
        right_on=target_time_col,
        how="inner",
        validate="one_to_one",
    )

    return {
        **raw_pair,
        "df_input": df_input,
        "df_target": df_target,
        "merged_df": merged_df,
        "input_audit": input_audit,
        "target_audit": target_audit,
        "alignment": alignment,
    }


def load_and_validate_pair(park_id: str) -> dict:
    """
    Convenience wrapper used for deterministic sample inspection and full audit execution.
    """
    raw_pair = load_pair_raw(park_id)
    return validate_loaded_pair(raw_pair)


def build_base_audit_row(park_id: str) -> dict:
    """
    Initialize a consistent audit row schema for every park.

    Keeping a fixed schema makes downstream summaries cleaner and failure reporting easier to interpret.
    """
    return {
        "park_id": park_id,
        "status": None,
        "failure_stage": None,
        "failure_reason": None,
        "input_file": input_files.get(park_id, Path("")).name or None,
        "target_file": target_files.get(park_id, Path("")).name or None,
        "input_sep_used": None,
        "target_sep_used": None,
        "input_time_col": None,
        "target_time_col": None,
        "input_rows": None,
        "target_rows": None,
        "merged_rows": None,
        "input_nat_count": None,
        "target_nat_count": None,
        "input_duplicate_count": None,
        "target_duplicate_count": None,
        "input_monotonic_before_sort": None,
        "target_monotonic_before_sort": None,
        "input_min_time": None,
        "input_max_time": None,
        "target_min_time": None,
        "target_max_time": None,
        "left_only_count": None,
        "right_only_count": None,
        "exact_timestamp_match": None,
    }


def audit_pair_with_logging(park_id: str) -> dict:
    """
    Audit one park while preserving interpretable diagnostics even when validation fails.

    The function separates:
    - pairing failures,
    - raw loading / separator / schema failures,
    - timestamp parsing failures,
    - duplicate timestamp failures,
    - merge validation failures.
    """
    row = build_base_audit_row(park_id)

    # Pairing failures are logged explicitly before any file loading is attempted.
    if park_id not in input_files:
        row["status"] = "failed"
        row["failure_stage"] = "pairing"
        row["failure_reason"] = "Missing input file for this park ID."
        return row

    if park_id not in target_files:
        row["status"] = "failed"
        row["failure_stage"] = "pairing"
        row["failure_reason"] = "Missing target file for this park ID."
        return row

    # First stage: raw loading, separator resolution, and timestamp-column resolution.
    try:
        raw_pair = load_pair_raw(park_id)
        row["input_sep_used"] = raw_pair["input_sep_used"]
        row["target_sep_used"] = raw_pair["target_sep_used"]
        row["input_time_col"] = raw_pair["input_time_col"]
        row["target_time_col"] = raw_pair["target_time_col"]

    except Exception as exc:
        row["status"] = "failed"
        message = str(exc)

        if "Separator detection failed" in message:
            row["failure_stage"] = "separator"
        elif "timestamp column" in message.lower():
            row["failure_stage"] = "schema"
        else:
            row["failure_stage"] = "raw_loading"

        row["failure_reason"] = message
        return row

    # Second stage: strict timestamp validation and merge integrity checks.
    try:
        result = validate_loaded_pair(raw_pair)

        row.update(
            {
                "input_rows": result["input_audit"]["row_count"],
                "target_rows": result["target_audit"]["row_count"],
                "merged_rows": len(result["merged_df"]),
                "input_nat_count": result["input_audit"]["nat_count"],
                "target_nat_count": result["target_audit"]["nat_count"],
                "input_duplicate_count": result["input_audit"]["duplicate_count"],
                "target_duplicate_count": result["target_audit"]["duplicate_count"],
                "input_monotonic_before_sort": result["input_audit"]["monotonic_before_sort"],
                "target_monotonic_before_sort": result["target_audit"]["monotonic_before_sort"],
                "input_min_time": result["input_audit"]["min_time"],
                "input_max_time": result["input_audit"]["max_time"],
                "target_min_time": result["target_audit"]["min_time"],
                "target_max_time": result["target_audit"]["max_time"],
                "left_only_count": result["alignment"]["left_only_count"],
                "right_only_count": result["alignment"]["right_only_count"],
                "exact_timestamp_match": result["alignment"]["exact_match"],
            }
        )

        # A park is marked as warning only when raw ordering is not monotonic before sorting.
        if (
            result["input_audit"]["monotonic_before_sort"] and
            result["target_audit"]["monotonic_before_sort"]
        ):
            row["status"] = "ok"
        else:
            row["status"] = "warning"
            row["failure_stage"] = "ordering"
            row["failure_reason"] = (
                "Raw rows were not fully monotonic before canonical sorting, "
                "but strict parsing and exact timestamp alignment succeeded."
            )

        return row

    except Exception as exc:
        message = str(exc)

        # Try to preserve partial diagnostics even if final validation fails.
        try:
            df_input = raw_pair["df_input"].copy()
            df_target = raw_pair["df_target"].copy()

            df_input[raw_pair["input_time_col"]] = parse_timestamp_strict(
                df_input[raw_pair["input_time_col"]],
                INPUT_TS_FORMATS,
                field_name=f"{raw_pair['input_path'].name}:{raw_pair['input_time_col']}",
            )
            df_target[raw_pair["target_time_col"]] = parse_timestamp_strict(
                df_target[raw_pair["target_time_col"]],
                TARGET_TS_FORMATS,
                field_name=f"{raw_pair['target_path'].name}:{raw_pair['target_time_col']}",
            )

            input_audit = audit_time_series(df_input[raw_pair["input_time_col"]])
            target_audit = audit_time_series(df_target[raw_pair["target_time_col"]])

            row.update(
                {
                    "input_rows": input_audit["row_count"],
                    "target_rows": target_audit["row_count"],
                    "input_nat_count": input_audit["nat_count"],
                    "target_nat_count": target_audit["nat_count"],
                    "input_duplicate_count": input_audit["duplicate_count"],
                    "target_duplicate_count": target_audit["duplicate_count"],
                    "input_monotonic_before_sort": input_audit["monotonic_before_sort"],
                    "target_monotonic_before_sort": target_audit["monotonic_before_sort"],
                    "input_min_time": input_audit["min_time"],
                    "input_max_time": input_audit["max_time"],
                    "target_min_time": target_audit["min_time"],
                    "target_max_time": target_audit["max_time"],
                }
            )

            # Alignment diagnostics are only meaningful if parsing and duplicate checks have already passed.
            if (
                input_audit["nat_count"] == 0 and
                target_audit["nat_count"] == 0 and
                input_audit["duplicate_count"] == 0 and
                target_audit["duplicate_count"] == 0
            ):
                alignment = validate_exact_timestamp_alignment(
                    df_input[raw_pair["input_time_col"]],
                    df_target[raw_pair["target_time_col"]],
                )
                row["left_only_count"] = alignment["left_only_count"]
                row["right_only_count"] = alignment["right_only_count"]
                row["exact_timestamp_match"] = alignment["exact_match"]

        except Exception:
            # The audit row should still be returned even if partial recovery fails.
            pass

        # Map the failure to an interpretable stage label.
        if "strict timestamp parsing failed" in message.lower() or "nat values" in message.lower():
            stage = "timestamp_parsing"
        elif "duplicate timestamps" in message.lower():
            stage = "duplicate_timestamp"
        elif "timestamp mismatch" in message.lower():
            stage = "merge_validation"
        else:
            stage = "validation"

        row["status"] = "failed"
        row["failure_stage"] = stage
        row["failure_reason"] = message
        return row

In [26]:
# Inspect one deterministic sample pair before launching the full multi-park audit.
# The sample inspection is intentionally non-blocking so that the notebook can still
# proceed to the full audit even if the first paired park fails strict validation.
sample_park_id = paired_park_ids[0]

try:
    sample_result = load_and_validate_pair(sample_park_id)

    print(f"Sample park: {sample_park_id}")
    print(f"Input file : {sample_result['input_path'].name} | sep='{sample_result['input_sep_used']}'")
    print(f"Target file: {sample_result['target_path'].name} | sep='{sample_result['target_sep_used']}'")
    print(f"Input time column : {sample_result['input_time_col']}")
    print(f"Target time column: {sample_result['target_time_col']}")

    print("\nInput columns:")
    print(sample_result["df_input"].columns.tolist()[:20])

    print("\nTarget columns:")
    print(sample_result["df_target"].columns.tolist())

    print("\nSample input preview:")
    display(sample_result["df_input"].head())

    print("\nSample target preview:")
    display(sample_result["df_target"].head())

except Exception as exc:
    sample_result = None
    print(f"Sample inspection skipped for park {sample_park_id}.")
    print(f"Reason: {exc}")

Sample park: 00011
Input file : data_input_00011.csv | sep=';'
Target file: data_target_00011.csv | sep=','
Input time column : fcst_time
Target time column: time

Input columns:
['fcst_time', 'nwp_fcst_horiz_hours', 'T_HAG_2_M', 'RELHUM_HAG_2_M', 'PS_SFC_0_M', 'U_GVL_58_HL', 'V_GVL_58_HL', 'U_GVL_60_HL', 'V_GVL_60_HL', 'ASWDIFDS_SFC_0_M', 'ASWDIRS_SFC_0_M', 'U_GVL_58_HL_m1', 'V_GVL_58_HL_m1', 'U_GVL_60_HL_m1', 'V_GVL_60_HL_m1', 'U_GVL_58_HL_p1', 'V_GVL_58_HL_p1', 'U_GVL_60_HL_p1', 'V_GVL_60_HL_p1']

Target columns:
['time', 'test_flag', 'pw', 'icon_eu_daf_pc_baseline']

Sample input preview:


,fcst_time,nwp_fcst_horiz_hours,T_HAG_2_M,RELHUM_HAG_2_M,PS_SFC_0_M,U_GVL_58_HL,V_GVL_58_HL,U_GVL_60_HL,V_GVL_60_HL,ASWDIFDS_SFC_0_M,ASWDIRS_SFC_0_M,U_GVL_58_HL_m1,V_GVL_58_HL_m1,U_GVL_60_HL_m1,V_GVL_60_HL_m1,U_GVL_58_HL_p1,V_GVL_58_HL_p1,U_GVL_60_HL_p1,V_GVL_60_HL_p1
0,2018-12-08 00:00:00,24,277105,79144,93060656,11219,-1154,6280,-324,25250,15902,12240,-1231,6980,-473,10205,-2168,5744,-1034
1,2018-12-08 01:00:00,25,276581,80629,93124426,12240,-1231,6980,-473,24238,15266,11536,346,6644,336,11219,-1154,6280,-324
2,2018-12-08 02:00:00,26,276119,81829,93155008,11536,346,6644,336,23307,14680,12284,1659,7124,1062,12240,-1231,6980,-473
3,2018-12-08 03:00:00,27,275763,80457,93182648,12284,1659,7124,1062,22443,14137,12939,968,7530,627,11536,346,6644,336
4,2018-12-08 04:00:00,28,275410,79750,93234324,12939,968,7530,627,21643,13629,11216,2137,6384,1384,12284,1659,7124,1062



Sample target preview:


,time,test_flag,pw,icon_eu_daf_pc_baseline
0,2018-12-08 00:00:00,0,0.100,0.806
1,2018-12-08 01:00:00,0,0.126,0.895
2,2018-12-08 02:00:00,0,0.232,0.829
3,2018-12-08 03:00:00,0,0.248,0.901
4,2018-12-08 04:00:00,0,0.232,0.937


In [27]:
# Summarize the sample park audit in a compact, thesis-ready table.
# The summary is only shown when the deterministic sample pair passes strict validation.
if sample_result is not None:
    sample_summary = pd.DataFrame(
        [
            {
                "park_id": sample_result["park_id"],
                "input_rows": sample_result["input_audit"]["row_count"],
                "target_rows": sample_result["target_audit"]["row_count"],
                "merged_rows": len(sample_result["merged_df"]),
                "input_nat_count": sample_result["input_audit"]["nat_count"],
                "target_nat_count": sample_result["target_audit"]["nat_count"],
                "input_duplicate_count": sample_result["input_audit"]["duplicate_count"],
                "target_duplicate_count": sample_result["target_audit"]["duplicate_count"],
                "input_monotonic_before_sort": sample_result["input_audit"]["monotonic_before_sort"],
                "target_monotonic_before_sort": sample_result["target_audit"]["monotonic_before_sort"],
                "input_min_time": sample_result["input_audit"]["min_time"],
                "input_max_time": sample_result["input_audit"]["max_time"],
                "target_min_time": sample_result["target_audit"]["min_time"],
                "target_max_time": sample_result["target_audit"]["max_time"],
                "left_only_count": sample_result["alignment"]["left_only_count"],
                "right_only_count": sample_result["alignment"]["right_only_count"],
                "exact_timestamp_match": sample_result["alignment"]["exact_match"],
            }
        ]
    )

    display(sample_summary)

    print("\nMerged sample preview:")
    display(sample_result["merged_df"].head())
else:
    print("Sample summary was skipped because the deterministic sample pair did not pass strict validation.")

,park_id,input_rows,target_rows,merged_rows,input_nat_count,target_nat_count,input_duplicate_count,target_duplicate_count,input_monotonic_before_sort,target_monotonic_before_sort,input_min_time,input_max_time,target_min_time,target_max_time,left_only_count,right_only_count,exact_timestamp_match
0,00011,12430,12430,12430,0,0,0,0,True,True,2018-12-08,2020-06-01 23:00:00,2018-12-08,2020-06-01 23:00:00,0,0,True



Merged sample preview:


,fcst_time,nwp_fcst_horiz_hours,T_HAG_2_M,RELHUM_HAG_2_M,PS_SFC_0_M,U_GVL_58_HL,V_GVL_58_HL,U_GVL_60_HL,V_GVL_60_HL,ASWDIFDS_SFC_0_M,...,U_GVL_60_HL_m1,V_GVL_60_HL_m1,U_GVL_58_HL_p1,V_GVL_58_HL_p1,U_GVL_60_HL_p1,V_GVL_60_HL_p1,time,test_flag,pw,icon_eu_daf_pc_baseline
0,2018-12-08 00:00:00,24,277105,79144,93060656,11219,-1154,6280,-324,25250,...,6980,-473,10205,-2168,5744,-1034,2018-12-08 00:00:00,0,0.100,0.806
1,2018-12-08 01:00:00,25,276581,80629,93124426,12240,-1231,6980,-473,24238,...,6644,336,11219,-1154,6280,-324,2018-12-08 01:00:00,0,0.126,0.895
2,2018-12-08 02:00:00,26,276119,81829,93155008,11536,346,6644,336,23307,...,7124,1062,12240,-1231,6980,-473,2018-12-08 02:00:00,0,0.232,0.829
3,2018-12-08 03:00:00,27,275763,80457,93182648,12284,1659,7124,1062,22443,...,7530,627,11536,346,6644,336,2018-12-08 03:00:00,0,0.248,0.901
4,2018-12-08 04:00:00,28,275410,79750,93234324,12939,968,7530,627,21643,...,6384,1384,12284,1659,7124,1062,2018-12-08 04:00:00,0,0.232,0.937


In [28]:
# Execute the strict audit across all discovered park IDs.
# This includes:
# - fully paired parks,
# - parks with missing input files,
# - parks with missing target files.
audit_rows = []

for park_id in tqdm(all_park_ids, desc="Strict raw audit"):
    audit_rows.append(audit_pair_with_logging(park_id))

audit_df = pd.DataFrame(audit_rows)

display(audit_df.head())

Strict raw audit:   0%|          | 0/272 [00:00<?, ?it/s]

,park_id,status,failure_stage,failure_reason,input_file,target_file,input_sep_used,target_sep_used,input_time_col,target_time_col,...,target_duplicate_count,input_monotonic_before_sort,target_monotonic_before_sort,input_min_time,input_max_time,target_min_time,target_max_time,left_only_count,right_only_count,exact_timestamp_match
0,00011,ok,None,None,data_input_00011.csv,data_target_00011.csv,;,",",fcst_time,time,...,0,True,True,2018-12-08 00:00:00,2020-06-01 23:00:00,2018-12-08 00:00:00,2020-06-01 23:00:00,0,0,True
1,00090,ok,None,None,data_input_00090.csv,data_target_00090.csv,",",",",fcst_time,time,...,0,True,True,2018-12-08 00:00:00,2020-06-01 23:00:00,2018-12-08 00:00:00,2020-06-01 23:00:00,0,0,True
2,00096,ok,None,None,data_input_00096.csv,data_target_00096.csv,",",",",fcst_time,time,...,0,True,True,2019-04-09 12:00:00,2020-06-01 23:00:00,2019-04-09 12:00:00,2020-06-01 23:00:00,0,0,True
3,00161,ok,None,None,data_input_00161.csv,data_target_00161.csv,",",",",fcst_time,time,...,0,True,True,2018-12-08 00:00:00,2020-06-01 23:00:00,2018-12-08 00:00:00,2020-06-01 23:00:00,0,0,True
4,00164,ok,None,None,data_input_00164.csv,data_target_00164.csv,",",",",fcst_time,time,...,0,True,True,2018-12-08 00:00:00,2020-06-01 23:00:00,2018-12-08 00:00:00,2020-06-01 23:00:00,0,0,True


In [29]:
# Aggregate the park-level audit outcomes into thesis-friendly summary views.
status_counts = (
    audit_df["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="parks")
)
display(status_counts)

# Partition the audit table by status for easier downstream inspection.
ok_df = audit_df[audit_df["status"] == "ok"].copy()
warning_df = audit_df[audit_df["status"] == "warning"].copy()
failed_df = audit_df[audit_df["status"] == "failed"].copy()

print(f"OK parks      : {len(ok_df)}")
print(f"Warning parks : {len(warning_df)}")
print(f"Failed parks  : {len(failed_df)}")

print("\nPairing status summary:")
display(
    pair_audit_df["pair_status"]
    .value_counts(dropna=False)
    .rename_axis("pair_status")
    .reset_index(name="parks")
)

# Summarize separator usage among audited parks to make parsing assumptions transparent.
if not audit_df.empty:
    print("\nSeparator usage among audited pairs:")
    display(
        audit_df.groupby(["input_sep_used", "target_sep_used"], dropna=False)
        .size()
        .reset_index(name="parks")
        .sort_values("parks", ascending=False)
        .reset_index(drop=True)
    )

# Report the validated global timestamp range only for parks that passed strict parsing and merge checks.
valid_df = audit_df[audit_df["status"].isin(["ok", "warning"])].copy()

if not valid_df.empty:
    print("\nGlobal validated timestamp range:")
    print("Input :", valid_df["input_min_time"].min(), "->", valid_df["input_max_time"].max())
    print("Target:", valid_df["target_min_time"].min(), "->", valid_df["target_max_time"].max())

# Show parks where raw ordering was imperfect but final alignment still succeeded.
if not warning_df.empty:
    print("\nOrdering warnings:")
    display(
        warning_df[
            [
                "park_id",
                "input_monotonic_before_sort",
                "target_monotonic_before_sort",
                "failure_reason",
            ]
        ].head(20)
    )

# Show failed parks with interpretable stage labels and reasons.
if not failed_df.empty:
    print("\nFailed parks:")
    display(
        failed_df[
            [
                "park_id",
                "failure_stage",
                "failure_reason",
                "input_file",
                "target_file",
            ]
        ].head(20)
    )

    print("\nTop failure reasons:")
    display(
        failed_df.groupby(["failure_stage", "failure_reason"], dropna=False)
        .size()
        .reset_index(name="parks")
        .sort_values("parks", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

,status,parks
0,ok,272


OK parks      : 272
Warning parks : 0
Failed parks  : 0

Pairing status summary:


,pair_status,parks
0,paired,272



Separator usage among audited pairs:


,input_sep_used,target_sep_used,parks
0,",",",",271
1,;,",",1



Global validated timestamp range:
Input : 2018-01-25 00:00:00 -> 2020-06-01 23:00:00
Target: 2018-01-25 00:00:00 -> 2020-06-01 23:00:00


## Συμπέρασμα

Το NB02 λειτουργεί ως **strict raw validation gate** πριν από οποιοδήποτε downstream notebook.

Επιβεβαιώνει:
- το αυστηρό input-target file pairing,
- το separator handling ανά αρχείο,
- το **strict timestamp parsing** με explicit allowed formats,
- την temporal ordering integrity ανά park,
- και το **exact input-target timestamp alignment**.

Άρα το notebook παραμένει μεθοδολογικά καθαρό:
- χωρίς modeling,
- χωρίς splitting,
- χωρίς feature engineering,
- και χωρίς claims πέρα από raw decoding / timestamp integrity validation.

Practical note:
Το τρέχον local raw directory αποδίδει 272 πλήρως ζευγοποιημένα input-target park pairs.
Αυτό θα πρέπει να ελεγχθεί αργότερα έναντι της documentation του πλήρους DaKS wind dataset, το οποίο στη δημοσίευση αναφέρεται ως 273 wind plants.